# Transformer from Scratch: Positional Encoding Benchmarks
Dataset: **Tiny Shakespeare** (character-level). Model: decoder-only Transformer from *Attention Is All You Need*.

We compare four ways of giving the model a sense of position:

| Name | Idea |
|---|---|
| `sinusoidal` | Static sin/cos vectors added to the embeddings (original paper) |
| `learned` | Trainable vector per position, added to the embeddings |
| `rope` | Rotary embedding: rotates Q and K inside attention (relative position) |
| `none` | No position information (ablation baseline) |

All models share the same seed, hyperparameters, and data split. The code lives in `transformer_lib.py`; this notebook only runs experiments and shows results.

In [ ]:
# !pip install torch matplotlib pandas   # uncomment if needed
import torch, pandas as pd, matplotlib.pyplot as plt
from transformer_lib import Config, POS_TYPES, load_data, train, evaluate_at_length, save_results
print("CUDA available:", torch.cuda.is_available())

## 1. Configuration
Lower `max_iters` (e.g. 300) for a quick test; use 3000+ for meaningful results.

In [ ]:
cfg = Config(
    max_iters=3000,
    eval_interval=250,
    # d_model=256, n_heads=4, n_layers=4, block_size=128 are the defaults
)
print(cfg)

## 2. Load data

In [ ]:
train_data, val_data, chars, stoi, itos = load_data()
vocab_size = len(chars)
print(f"vocab size: {vocab_size} | train chars: {len(train_data):,} | val chars: {len(val_data):,}")

## 3. Train all four variants

In [ ]:
results = {}
for pos in POS_TYPES:
    results[pos] = train(pos, cfg, train_data, val_data, vocab_size, itos)
save_results(results, "results.json")

## 4. Loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for pos, r in results.items():
    h = r["history"]
    axes[0].plot(h["step"], h["train"], label=pos)
    axes[1].plot(h["step"], h["val"], label=pos)
axes[0].set_title("Training loss"); axes[1].set_title("Validation loss")
for ax in axes:
    ax.set_xlabel("step"); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel("cross-entropy loss")
plt.tight_layout(); plt.savefig("loss_curves.png", dpi=150); plt.show()

## 5. Summary table
Final loss, parameter count, and training speed. Perplexity = exp(val loss).

In [ ]:
rows = []
for pos, r in results.items():
    h = r["history"]
    rows.append({
        "encoding": pos,
        "final val loss": h["val"][-1],
        "best val loss": min(h["val"]),
        "val perplexity": torch.exp(torch.tensor(h["val"][-1])).item(),
        "params (M)": r["params"] / 1e6,
        "train time (s)": r["train_time_s"],
        "ms / step": r["sec_per_step"] * 1000,
    })
summary = pd.DataFrame(rows).set_index("encoding").round(4)
summary.sort_values("final val loss")

## 6. Length extrapolation
Models were trained with context length `block_size`. How do they do on **longer** sequences they never saw?
`learned` has no vectors for unseen positions, so it is reported as NaN (cannot run).

In [ ]:
lengths = [cfg.block_size, cfg.block_size * 2, cfg.block_size * 4]
extra = {pos: [evaluate_at_length(r["model"], val_data, L, cfg) for L in lengths] for pos, r in results.items()}
extra_df = pd.DataFrame(extra, index=[f"{L} tokens" for L in lengths]).T.round(4)
extra_df

In [ ]:
plt.figure(figsize=(7, 4.5))
for pos, vals in extra.items():
    plt.plot(lengths, vals, marker="o", label=pos)
plt.axvline(cfg.block_size, color="gray", ls="--", label="training length")
plt.xlabel("evaluation sequence length"); plt.ylabel("validation loss")
plt.title("Generalisation to longer sequences"); plt.grid(alpha=0.3); plt.legend()
plt.tight_layout(); plt.savefig("extrapolation.png", dpi=150); plt.show()

## 7. Generated samples

In [ ]:
for pos, r in results.items():
    print(f"=== {pos} ===")
    print(r["sample"][:400])
    print()

## 8. Conclusions
*Fill this in after running.* Suggested points to discuss:
- Which encoding reached the lowest validation loss, and by how much? Is the gap larger than run-to-run noise (try several seeds)?
- How much worse is `none`? This shows how much position information the causal mask alone provides.
- Parameter cost: `learned` adds `block_size x d_model` parameters; the others add none.
- Speed: does RoPE's extra rotation in every layer cost measurable time per step?
- Extrapolation: which encodings degrade gracefully beyond the training length?